In [17]:
import pyodbc
import pandas as pd
import numpy as np

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'B1CRHPROXY', 'ODBC Driver 13 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [43]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv()

SERVER = os.getenv("DB_SERVER1")
DBNAME = os.getenv("DB_NAME1")
USER = os.getenv("DB_USER1")
PASS = os.getenv("DB_PASSWORD1")
ENCRYPT = os.getenv("DB_ENCRYPT", "yes")
TRUST = os.getenv("DB_TRUST_CERT", "yes")

SERVER = SERVER.replace("\\\\", "\\")  # normaliza


cnx = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DBNAME};"
    f"UID={USER};"
    f"PWD={PASS};"
    f"Encrypt={ENCRYPT};TrustServerCertificate={TRUST};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(cnx)}")

In [44]:
from sqlalchemy import text
try:
    with engine.connect() as conn:
        print("Ping:", conn.execute(text("SELECT 1")).scalar_one())
        print("Version:", conn.execute(text("SELECT @@VERSION")).scalar_one())
        print("✅ Conexión OK")
except Exception as e:
    print("❌ Error al conectar:", e)

Ping: 1
Version: Microsoft SQL Server 2017 (RTM) - 14.0.1000.169 (X64) 
	Aug 22 2017 17:04:49 
	Copyright (C) 2017 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2019 Standard 10.0 <X64> (Build 17763: )

✅ Conexión OK


In [45]:
import pandas as pd
# 🔥 TEST 2: traer datos a pandas
query = "SELECT TOP 10 * FROM OITM"

df = pd.read_sql(query, engine)

df.head(5)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod
0,"# 111A-30"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
1,"# 111A-30"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
2,"# 111A-54"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
3,"# 111A-54"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
4,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None


In [50]:
query = """
WITH Precios AS (
    SELECT
        T0.ItemCode,

        MAX(CASE WHEN T0.PriceList = 14 THEN T0.Price END) AS Precio_B2B_Estandar,
        MAX(CASE WHEN T0.PriceList = 12 THEN T0.Price END) AS Precio_B2B_VIP,
        MAX(CASE WHEN T0.PriceList = 13 THEN T0.Price END) AS Precio_B2C,
        MAX(CASE WHEN T0.PriceList = 11 THEN T0.Price END) AS Precio_Promociones_Ecommerce

    FROM ITM1 T0
    WHERE
        T0.PriceList IN (11, 12, 13, 14)
    GROUP BY
        T0.ItemCode
)

SELECT TOP 10

    -- ============================================================
    -- IDENTIFICACIÓN
    -- ============================================================
    T0.ItemCode AS Codigo_Articulo,
    T0.ItemName AS Descripcion_Completa,
    T0.ItmsGrpCod AS Codigo_Grupo_Articulo,
    T1.ItmsGrpNam AS Grupo_Articulo,
    T0.InvntryUom AS Unidad_Medida_Inventario,
    T0.FirmCode AS Codigo_Fabricante,
    T2.FirmName AS Marca_Fabricante,

    -- ============================================================
    -- COMPRAS / PROVEEDOR
    -- ============================================================
    T0.CardCode AS Codigo_Proveedor_Principal,
    T3.CardName AS Proveedor_Principal,
    T0.LastPurDat AS Ultima_Fecha_Compra,
    T0.LastPurPrc AS Ultimo_Precio_Compra,

    -- ============================================================
    -- COSTOS
    -- ============================================================
    T0.AvgPrice AS Costo_Promedio_Articulo,
    T4.AvgPrice AS Costo_Promedio_Bodega,

    -- ============================================================
    -- INVENTARIO POR BODEGA
    -- ============================================================
    T4.WhsCode AS Codigo_Bodega,
    T4.OnHand AS Stock_En_Mano_Bodega,
    T4.IsCommited AS Stock_Comprometido_Bodega,
    T4.OnOrder AS Stock_En_Pedido_Transito_Bodega,
    T4.MinStock AS Stock_Minimo_Bodega,
    T4.MaxStock AS Stock_Maximo_Bodega,

    T4.OnHand - T4.IsCommited + T4.OnOrder AS Stock_Disponible_Teorico,

    -- ============================================================
    -- PRECIOS POR LISTA
    -- ============================================================
    P.Precio_B2B_Estandar,
    P.Precio_B2B_VIP,
    P.Precio_B2C,
    P.Precio_Promociones_Ecommerce

FROM OITM T0

LEFT JOIN OITB T1
    ON T0.ItmsGrpCod = T1.ItmsGrpCod

LEFT JOIN OMRC T2
    ON T0.FirmCode = T2.FirmCode

LEFT JOIN OCRD T3
    ON T0.CardCode = T3.CardCode

LEFT JOIN OITW T4
    ON T0.ItemCode = T4.ItemCode

LEFT JOIN Precios P
    ON T0.ItemCode = P.ItemCode

WHERE
    T0.SellItem = 'Y'
    AND T0.PrchseItem = 'Y'
    AND T0.InvntItem = 'Y'
    AND T0.validFor = 'Y'
    AND T0.frozenFor = 'N'
    AND T4.WhsCode IS NOT NULL
    AND T2.FirmName LIKE '%SIEMENS%'

ORDER BY
    T0.ItemCode,
    T4.WhsCode;
"""

In [51]:
df = pd.read_sql(query, engine)

df.head(510)

,Codigo_Articulo,Descripcion_Completa,Codigo_Grupo_Articulo,Grupo_Articulo,Unidad_Medida_Inventario,Codigo_Fabricante,Marca_Fabricante,Codigo_Proveedor_Principal,Proveedor_Principal,Ultima_Fecha_Compra,...,Stock_En_Mano_Bodega,Stock_Comprometido_Bodega,Stock_En_Pedido_Transito_Bodega,Stock_Minimo_Bodega,Stock_Maximo_Bodega,Stock_Disponible_Teorico,Precio_B2B_Estandar,Precio_B2B_VIP,Precio_B2C,Precio_Promociones_Ecommerce
0,.50HP18RB-SIE-S,MOTOR MONOFASICO 120V 0.5HP 1800RPM A7B1000000...,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-02-22,...,0.0,0.0,0.0,0.0,0.0,0.0,106.00,100.00,118.0,0.0
1,.50HP18RB-SIE-S,MOTOR MONOFASICO 120V 0.5HP 1800RPM A7B1000000...,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-02-22,...,0.0,0.0,0.0,0.0,0.0,0.0,106.00,100.00,118.0,0.0
2,.50HP18RB-SIE-S,MOTOR MONOFASICO 120V 0.5HP 1800RPM A7B1000000...,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-02-22,...,0.0,0.0,0.0,NaN,NaN,0.0,106.00,100.00,118.0,0.0
3,.75HP18RB-SIE-S,MOTOR WEG 0.75HP 3PH 4P IP55 B35 71 IEC IE1,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,NaN,NaN,2022-04-22,...,0.0,0.0,0.0,0.0,0.0,0.0,258.00,244.00,287.0,0.0
4,.75HP18RB-SIE-S,MOTOR WEG 0.75HP 3PH 4P IP55 B35 71 IEC IE1,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,NaN,NaN,2022-04-22,...,0.0,0.0,0.0,0.0,0.0,0.0,258.00,244.00,287.0,0.0
5,.75HP18RB-SIE-S,MOTOR WEG 0.75HP 3PH 4P IP55 B35 71 IEC IE1,125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,NaN,NaN,2022-04-22,...,0.0,0.0,0.0,NaN,NaN,0.0,258.00,244.00,287.0,0.0
6,01.0093,"CODO DE COBRE SOLDABLE DE 1-5/8"" A 90 GRADOS",105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-11-15,...,0.0,0.0,0.0,0.0,0.0,0.0,5.60,5.30,6.2,0.0
7,07.0001,TERMINAL TIPO BANDERA PARA ALAMBRE 12-10GA FD32C,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-11-15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.20,0.20,0.2,0.0
8,07.0021,CONTACTOR 40AMP 2POLOS 24V QCA-402 QUALITY,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2021-11-15,...,0.0,0.0,0.0,0.0,0.0,0.0,8.65,8.15,9.6,0.0
9,1.0HP18RB-SIE,"MOT.TRIFASICO 230/460V 1HP 1800RPM 143T GP100,...",125,MOTORES ELECTRIC. DE,UNI,39,SIEMENS,PNA0113,"SIEMENS, S.A.",2021-04-30,...,0.0,0.0,0.0,0.0,0.0,0.0,210.00,198.00,233.0,0.0


In [53]:
ruta_query_sap="./Datos/Prueba_campos_maestros_SAP.csv"
df.to_csv(ruta_query_sap)